# Exercise 3: Panel forecasting with traditional and text data

This notebook is designed as a simple macro forecasting benchmark.
The target is binary and the model is intentionally modest: a rolling logistic regression.


## Safe baseline before adding text

For the baseline, use only these traditional variables:

- `industrial_production_yoy`
- `inflation_yoy`
- `credit_growth_yoy`
- `policy_rate_change_bps`

Avoid using these columns as predictors in the benchmark:

- `recession_next_quarter` because it is the target
- `recession_current` because it would act like a persistence shortcut

The out-of-sample exercise will produce **one-step-ahead predictions from 2021 onward** for all countries.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from panelsplit.cross_validation import PanelSplit
from panelsplit.plot import plot_splits

candidates = [Path.cwd(), Path.cwd() / 'sessions' / 'in_class_assignment']
assignment_root = next(path for path in candidates if (path / 'data').exists())
data_dir = assignment_root / 'data' / 'synthetic_panel'

headlines_df = pd.read_csv(data_dir / 'synthetic_panel_headlines.csv')
features_df = pd.read_csv(data_dir / 'synthetic_panel_features.csv')
features_df['month'] = pd.to_datetime(features_df['month'])

allowed_features = [
    'industrial_production_yoy',
    'inflation_yoy',
    'credit_growth_yoy',
    'policy_rate_change_bps',
]
target_col = 'recession_next_quarter'

print('Headline corpus:', headlines_df.shape)
print('Country-month panel:', features_df.shape)
print('Allowed baseline features:', allowed_features)


## Where text could add predictive power

Students need to design their own text signals. Reasonable topics to extract include:

- weakening demand or exports
- tighter credit conditions
- layoffs, hiring freezes, or weaker labor demand
- inflation and cost-of-living pressure
- worsening business confidence or policy uncertainty

The goal is to add text features that make economic sense and are available at forecast time.


In [ ]:
corpus_preview = headlines_df.head(8)
print(corpus_preview.to_string(index=False))


In [ ]:
def add_recession_shading(ax, recession_months, color='0.85'):
    for month in recession_months:
        start = pd.Timestamp(month)
        end = start + pd.offsets.MonthEnd(1)
        ax.axvspan(start, end, color=color, alpha=0.45, linewidth=0)

panel_monthly = (
    features_df.groupby('month')[['industrial_production_yoy', 'inflation_yoy', 'credit_growth_yoy', 'recession_current']]
    .mean()
    .reset_index()
)
panel_recession_months = panel_monthly.loc[panel_monthly['recession_current'] >= 0.5, 'month']

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(panel_monthly['month'], panel_monthly['industrial_production_yoy'], color='#2f6db3', linewidth=2)
add_recession_shading(axes[0], panel_recession_months)
axes[0].set_title('Average industrial production with recession shading')
axes[0].set_ylabel('Industrial production (y/y)')
axes[0].grid(alpha=0.3)

axes[1].plot(panel_monthly['month'], panel_monthly['inflation_yoy'], color='#d95f02', linewidth=2, label='Inflation')
axes[1].plot(panel_monthly['month'], panel_monthly['credit_growth_yoy'], color='#228b22', linewidth=2, label='Credit growth')
add_recession_shading(axes[1], panel_recession_months)
axes[1].set_title('Other traditional features with the same recession shading')
axes[1].set_ylabel('Rate / growth')
axes[1].grid(alpha=0.3)
axes[1].legend(loc='upper right')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
case_country = (
    features_df.groupby('country')['recession_current']
    .mean()
    .sort_values(ascending=False)
    .index[0]
)
case_df = features_df.loc[features_df['country'] == case_country].sort_values('month')
case_recession_months = case_df.loc[case_df['recession_current'] == 1, 'month']

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(case_df['month'], case_df['industrial_production_yoy'], color='#2f6db3', linewidth=2, label='Industrial production (y/y)')
ax.plot(case_df['month'], case_df['credit_growth_yoy'], color='#228b22', linewidth=2, label='Credit growth (y/y)')
add_recession_shading(ax, case_recession_months)
ax.set_title(f'{case_country}: traditional features and recession dates')
ax.set_ylabel('Growth rate')
ax.grid(alpha=0.3)
ax.legend(loc='upper right')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
def prepare_panel_data(df: pd.DataFrame, feature_cols: list[str], target_col: str) -> pd.DataFrame:
    out = df.sort_values(['month', 'country']).copy()
    out = out.dropna(subset=feature_cols + [target_col])
    return out[['country', 'month'] + feature_cols + [target_col, 'recession_current']]

panel_df = prepare_panel_data(features_df, allowed_features, target_col)
panel_df[['country', 'month'] + allowed_features + [target_col]].head()


In [ ]:
splitter = PanelSplit(periods=panel_df['month'], n_splits=60, test_size=1)
plot_splits(splitter, show=True)


## Rolling one-step-ahead logistic benchmark

Each fold predicts one month ahead for all countries.
Since a single month may contain only one class in the target, it does **not** make sense to compute ROC-AUC fold by fold.
Instead, we stack all one-step-ahead predictions from 2021 onward and evaluate ROC-AUC and PR performance on the full forecast sample.


In [ ]:
splits = splitter.split()

prediction_rows = []
coefficient_rows = []
fold_rows = []

for fold_id, (train_idx, test_idx) in enumerate(splits, start=1):
    train_df = panel_df.iloc[train_idx].copy()
    test_df = panel_df.iloc[test_idx].copy()

    model = Pipeline(
        steps=[
            ('scaler', StandardScaler()),
            ('logit', LogisticRegression()),
        ]
    )
    model.fit(train_df[allowed_features], train_df[target_col])

    test_df['predicted_probability'] = model.predict_proba(test_df[allowed_features])[:, 1]
    test_df['predicted_class'] = (test_df['predicted_probability'] >= 0.5).astype(int)
    prediction_rows.append(test_df[['country', 'month', target_col, 'recession_current', 'predicted_probability', 'predicted_class']])

    fold_rows.append({
        'fold': fold_id,
        'train_end': train_df['month'].max().strftime('%Y-%m'),
        'test_month': test_df['month'].iloc[0].strftime('%Y-%m'),
        'test_positive_share': round(test_df[target_col].mean(), 4),
        'accuracy': round(accuracy_score(test_df[target_col], test_df['predicted_class']), 4),
        'precision': round(precision_score(test_df[target_col], test_df['predicted_class'], zero_division=0), 4),
        'recall': round(recall_score(test_df[target_col], test_df['predicted_class'], zero_division=0), 4),
    })

    coef = model.named_steps['logit'].coef_[0]
    coefficient_rows.append({'fold': fold_id, **{feature: value for feature, value in zip(allowed_features, coef)}})

forecast_df = pd.concat(prediction_rows, ignore_index=True)
fold_metrics = pd.DataFrame(fold_rows)
coef_df = pd.DataFrame(coefficient_rows)

print('Forecast period:', forecast_df['month'].min().strftime('%Y-%m'), 'to', forecast_df['month'].max().strftime('%Y-%m'))
fold_metrics.tail()


In [ ]:
overall_metrics = pd.DataFrame([
    {
        'roc_auc': round(roc_auc_score(forecast_df[target_col], forecast_df['predicted_probability']), 4),
        'pr_auc': round(average_precision_score(forecast_df[target_col], forecast_df['predicted_probability']), 4),
        'accuracy@0.5': round(accuracy_score(forecast_df[target_col], forecast_df['predicted_class']), 4),
        'precision@0.5': round(precision_score(forecast_df[target_col], forecast_df['predicted_class'], zero_division=0), 4),
        'recall@0.5': round(recall_score(forecast_df[target_col], forecast_df['predicted_class'], zero_division=0), 4),
        'brier': round(brier_score_loss(forecast_df[target_col], forecast_df['predicted_probability']), 4),
    }
])
overall_metrics


In [ ]:
fpr, tpr, _ = roc_curve(forecast_df[target_col], forecast_df['predicted_probability'])
precision, recall, _ = precision_recall_curve(forecast_df[target_col], forecast_df['predicted_probability'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(fpr, tpr, color='#2f6db3', linewidth=2, label=f"ROC-AUC = {overall_metrics.loc[0, 'roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
axes[0].set_title('ROC curve')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].grid(alpha=0.3)
axes[0].legend(loc='lower right')

axes[1].plot(recall, precision, color='#b22222', linewidth=2, label=f"PR-AUC = {overall_metrics.loc[0, 'pr_auc']:.3f}")
axes[1].set_title('Precision-recall curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(alpha=0.3)
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()


In [ ]:
feature_importance = (
    coef_df.drop(columns='fold')
    .abs()
    .mean()
    .sort_values(ascending=False)
    .rename('mean_abs_logit_coefficient')
    .reset_index()
    .rename(columns={'index': 'feature'})
)
feature_importance


In [ ]:
plt.figure(figsize=(8, 4))
plt.barh(feature_importance['feature'], feature_importance['mean_abs_logit_coefficient'], color='#2f6db3')
plt.gca().invert_yaxis()
plt.title('Average absolute coefficient across rolling logistic models')
plt.xlabel('Mean |coefficient|')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
case_forecast = forecast_df.loc[forecast_df['country'] == case_country].sort_values('month')
case_recession_months_forecast = case_df.loc[
    (case_df['month'] >= case_forecast['month'].min()) & (case_df['recession_current'] == 1),
    'month'
]

plt.figure(figsize=(11, 4))
plt.plot(case_forecast['month'], case_forecast['predicted_probability'], color='#2f6db3', linewidth=2, label='Predicted probability')
#plt.plot(case_forecast['month'], case_forecast[target_col], color='#b22222', linewidth=1.5, linestyle='--', label='Actual next-quarter recession')
ax = plt.gca()
add_recession_shading(ax, case_recession_months_forecast)
plt.title(f'{case_country}: one-step-ahead predicted recession probability')
plt.ylabel('Probability')
plt.grid(alpha=0.3)
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## What you can improve next

1. Replace the logistic benchmark with a more powerful model such as random forest, XGBoost, or CatBoost.
2. Build informative text features from the headline corpus and add them to the panel.
3. Tune the model inside the `PanelSplit` framework instead of keeping default hyperparameters.
4. Compare models with ROC-AUC and PR curves, not only with accuracy.
5. Show feature importance and discuss whether it is economically plausible.

The key discipline is always the same: keep the forecast evaluation out-of-sample and avoid leakage.
